# 🌫️ Delhi Air Quality Intelligence Dashboard
### Final Year Project — Run in Google Colab
---
**Just run all cells top to bottom. A public URL will appear at the end.**

In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install streamlit plotly pyngrok -q

In [ ]:
# ── Step 2: Write the dashboard file ─────────────────────────────────────────
dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Delhi Air Quality Intelligence", page_icon="\U0001f32b", layout="wide", initial_sidebar_state="expanded")

st.markdown(\'\'\'<style>
@import url(\'https://fonts.googleapis.com/css2?family=Rajdhani:wght@400;600;700&family=Exo+2:wght@300;400;600&display=swap\');
html, body, [class*="css"] { font-family: \'Exo 2\', sans-serif; background-color: #0a0e1a; color: #e0e6f0; }
.main { background: linear-gradient(135deg, #0a0e1a 0%, #0d1526 50%, #0a1020 100%); }
h1, h2, h3 { font-family: \'Rajdhani\', sans-serif; letter-spacing: 2px; }
.hero-banner { background: linear-gradient(135deg, #0d1b3e 0%, #1a0a2e 40%, #0d2b1a 100%); border: 1px solid rgba(255,100,50,0.3); border-radius: 16px; padding: 28px 36px; margin-bottom: 24px; }
.hero-title { font-family: \'Rajdhani\', sans-serif; font-size: 2.6rem; font-weight: 700; letter-spacing: 4px; background: linear-gradient(90deg, #ff6433, #ffb347, #ff4444); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin: 0; }
.hero-sub { color: #8899bb; font-size: 0.95rem; letter-spacing: 2px; margin-top: 6px; text-transform: uppercase; }
.kpi-card { background: linear-gradient(145deg, #111827, #0f1f35); border: 1px solid rgba(255,255,255,0.08); border-radius: 14px; padding: 20px 22px; text-align: center; }
.kpi-value { font-family: \'Rajdhani\', sans-serif; font-size: 2.4rem; font-weight: 700; color: #ff8c42; line-height: 1; }
.kpi-label { font-size: 0.75rem; letter-spacing: 2px; text-transform: uppercase; color: #667799; margin-top: 6px; }
.kpi-status { font-size: 0.8rem; margin-top: 8px; font-weight: 600; }
.section-header { font-family: \'Rajdhani\', sans-serif; font-size: 1.4rem; font-weight: 700; letter-spacing: 3px; text-transform: uppercase; color: #ff9955; border-left: 4px solid #ff6433; padding-left: 14px; margin: 28px 0 16px 0; }
.alert-red { background: rgba(255,50,50,0.12); border: 1px solid rgba(255,50,50,0.4); border-radius:10px; padding:14px 18px; margin:8px 0; }
.alert-orange { background: rgba(255,140,0,0.12); border: 1px solid rgba(255,140,0,0.4); border-radius:10px; padding:14px 18px; margin:8px 0; }
.alert-green { background: rgba(50,200,100,0.10); border: 1px solid rgba(50,200,100,0.35); border-radius:10px; padding:14px 18px; margin:8px 0; }
.tip-card { background: linear-gradient(135deg, #0f1e35, #1a2840); border: 1px solid rgba(100,180,255,0.2); border-radius: 12px; padding: 16px 20px; margin: 8px 0; }
.tip-icon { font-size: 1.4rem; margin-bottom: 6px; }
.tip-title { font-family:\'Rajdhani\',sans-serif; font-size:1.05rem; color:#7ecfff; font-weight:600; }
.tip-text { font-size:0.85rem; color:#aab8cc; margin-top:4px; line-height:1.5; }
section[data-testid="stSidebar"] { background: linear-gradient(180deg, #080e1c 0%, #0a1525 100%); border-right: 1px solid rgba(255,100,50,0.15); }
</style>\'\'\', unsafe_allow_html=True)

@st.cache_data
def generate_data():
    np.random.seed(42)
    stations = {
        "Anand Vihar": (28.6469, 77.3161), "ITO": (28.6292, 77.2410),
        "RK Puram": (28.5651, 77.1875), "Punjabi Bagh": (28.6724, 77.1323),
        "Dwarka": (28.5823, 77.0500), "Rohini": (28.7395, 77.1273),
        "Okhla": (28.5406, 77.2673), "Shadipur": (28.6517, 77.1593),
        "Lodhi Road": (28.5908, 77.2270), "Bawana": (28.7914, 77.0424),
        "Mundka": (28.6847, 77.0260), "Wazirpur": (28.7041, 77.1673),
    }
    intensity = {"Anand Vihar":1.45,"ITO":1.20,"RK Puram":0.95,"Punjabi Bagh":1.10,
                 "Dwarka":0.90,"Rohini":1.05,"Okhla":1.15,"Shadipur":1.08,
                 "Lodhi Road":0.88,"Bawana":1.50,"Mundka":1.30,"Wazirpur":1.18}
    start = datetime(2023,1,1)
    dates = [start + timedelta(hours=i) for i in range(365*24)]
    records = []
    for station,(lat,lon) in stations.items():
        mul = intensity[station]
        for dt in dates:
            month=dt.month; hour=dt.hour
            seasonal=1.0
            if month in [11,12,1]: seasonal=1.6
            elif month in [10,2]: seasonal=1.3
            elif month in [4,5,6]: seasonal=0.8
            elif month in [7,8]: seasonal=0.65
            diurnal=1.0
            if hour in [7,8,9]: diurnal=1.3
            elif hour in [18,19,20]: diurnal=1.2
            elif hour in [3,4,5]: diurnal=0.7
            base_pm25=85*mul*seasonal*diurnal
            noise=lambda s: np.random.normal(0,s)
            pm25=max(5,base_pm25+noise(25))
            pm10=max(10,pm25*1.6+noise(30))
            no2=max(5,40*mul*seasonal*diurnal+noise(12))
            so2=max(2,12*mul*seasonal+noise(5))
            co=max(0.2,1.8*mul*seasonal*diurnal+noise(0.5))
            o3=max(5,35+noise(15)-(pm25*0.1))
            nh3=max(1,10*mul*seasonal+noise(4))
            temp=15+15*np.sin((month-1)/12*2*np.pi)+noise(2)
            hum=min(100,max(20,60+20*np.cos((month-6)/12*2*np.pi)+noise(8)))
            def pm25_aqi(c):
                if c<=30: return c/30*50
                elif c<=60: return 50+(c-30)/30*50
                elif c<=90: return 100+(c-60)/30*100
                elif c<=120: return 200+(c-90)/30*100
                elif c<=250: return 300+(c-120)/130*100
                else: return 400+min(100,(c-250)/130*100)
            aqi=pm25_aqi(pm25)
            records.append({"datetime":dt,"station":station,"lat":lat,"lon":lon,
                "PM2.5":round(pm25,1),"PM10":round(pm10,1),"NO2":round(no2,1),
                "SO2":round(so2,1),"CO":round(co,2),"O3":round(o3,1),"NH3":round(nh3,1),
                "Temperature":round(temp,1),"Humidity":round(hum,1),"AQI":round(aqi)})
    df=pd.DataFrame(records)
    df["date"]=df["datetime"].dt.date
    df["month"]=df["datetime"].dt.month
    df["hour"]=df["datetime"].dt.hour
    df["month_name"]=df["datetime"].dt.strftime("%b")
    def aqi_category(a):
        if a<=50: return ("Good","#00e676")
        elif a<=100: return ("Satisfactory","#aeea00")
        elif a<=200: return ("Moderate","#ffeb3b")
        elif a<=300: return ("Poor","#ff9800")
        elif a<=400: return ("Very Poor","#f44336")
        else: return ("Severe","#7b1fa2")
    df["Category"],df["Cat_Color"]=zip(*df["AQI"].map(aqi_category))
    return df,stations

df,stations=generate_data()
pollutants=["PM2.5","PM10","NO2","SO2","CO","O3","NH3"]
months_order=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

with st.sidebar:
    st.markdown("### \U0001f32b DELHI AQI INTEL")
    st.markdown("---")
    selected_stations=st.multiselect("\U0001f4cd Monitoring Stations",options=list(stations.keys()),default=["Anand Vihar","ITO","RK Puram","Bawana"])
    selected_pollutant=st.selectbox("\U0001f9ea Primary Pollutant",pollutants,index=0)
    month_range=st.slider("\U0001f4c5 Month Range",1,12,(1,12))
    hour_range=st.slider("\U0001f550 Hour Range",0,23,(0,23))
    st.markdown("---")
    st.markdown("**AQI Scale Reference**")
    for label,color,rng in [("Good","#00e676","0-50"),("Satisfactory","#aeea00","51-100"),("Moderate","#ffeb3b","101-200"),("Poor","#ff9800","201-300"),("Very Poor","#f44336","301-400"),("Severe","#9c27b0","401-500")]:
        st.markdown(f\'<span style="color:{color}">\u25cf</span> **{label}** ({rng})\',unsafe_allow_html=True)

if not selected_stations: selected_stations=list(stations.keys())
mask=(df["station"].isin(selected_stations))&(df["month"].between(*month_range))&(df["hour"].between(*hour_range))
filtered=df[mask].copy()

avg_aqi=int(filtered["AQI"].mean())
avg_pm25=round(filtered["PM2.5"].mean(),1)
worst_st=filtered.groupby("station")["AQI"].mean().idxmax()
best_hour=int(filtered.groupby("hour")["AQI"].mean().idxmin())
cat_colors={"Good":"#00e676","Satisfactory":"#aeea00","Moderate":"#ffeb3b","Poor":"#ff9800","Very Poor":"#f44336","Severe":"#9c27b0"}
if avg_aqi<=50: cat,emoji="Good","\u2705"
elif avg_aqi<=100: cat,emoji="Satisfactory","\U0001f7e1"
elif avg_aqi<=200: cat,emoji="Moderate","\u26a0\ufe0f"
elif avg_aqi<=300: cat,emoji="Poor","\U0001f7e0"
elif avg_aqi<=400: cat,emoji="Very Poor","\U0001f534"
else: cat,emoji="Severe","\u2620\ufe0f"
cat_color=cat_colors.get(cat,"#ff6433")

st.markdown(f\'\'\'<div class="hero-banner"><p class="hero-title">\U0001f32b DELHI AIR QUALITY INTELLIGENCE</p><p class="hero-sub">Real-time Pollution Monitoring &middot; Forecasting &middot; Hotspot Analysis &middot; Health Advisory</p></div>\'\'\',unsafe_allow_html=True)
c1,c2,c3,c4=st.columns(4)
with c1: st.markdown(f\'<div class="kpi-card"><div class="kpi-value" style="color:{cat_color}">{avg_aqi}</div><div class="kpi-label">Average AQI</div><div class="kpi-status" style="color:{cat_color}">{emoji} {cat}</div></div>\',unsafe_allow_html=True)
with c2: st.markdown(f\'<div class="kpi-card"><div class="kpi-value">{avg_pm25}</div><div class="kpi-label">Avg PM2.5 (\u00b5g/m\u00b3)</div><div class="kpi-status" style="color:#ff8c42">{"\u26a0\ufe0f High" if avg_pm25>60 else "\u2705 OK"}</div></div>\',unsafe_allow_html=True)
with c3: st.markdown(f\'<div class="kpi-card"><div class="kpi-value" style="font-size:1.5rem;color:#ff4444">{worst_st}</div><div class="kpi-label">Most Polluted Station</div><div class="kpi-status" style="color:#ff4444">\U0001f534 Hotspot</div></div>\',unsafe_allow_html=True)
with c4: st.markdown(f\'<div class="kpi-card"><div class="kpi-value" style="color:#00e676">{best_hour:02d}:00</div><div class="kpi-label">Cleanest Hour</div><div class="kpi-status" style="color:#00e676">\U0001f33f Best to go out</div></div>\',unsafe_allow_html=True)

st.markdown("<br>",unsafe_allow_html=True)
tab1,tab2,tab3,tab4,tab5,tab6=st.tabs(["\U0001f4c8 Trend Analysis","\U0001f5fa Hotspot Map","\U0001f52e Forecast","\u23f0 Diurnal Patterns","\U0001f9ea Pollutant Breakdown","\U0001f4a1 Health Advisory"])

def aqi_cat(a):
    if a<=50: return "Good"
    elif a<=100: return "Satisfactory"
    elif a<=200: return "Moderate"
    elif a<=300: return "Poor"
    elif a<=400: return "Very Poor"
    else: return "Severe"

with tab1:
    st.markdown(\'<div class="section-header">Monthly Pollution Trend</div>\',unsafe_allow_html=True)
    monthly=filtered.groupby(["month","month_name","station"])[selected_pollutant].mean().reset_index().sort_values("month")
    fig=px.line(monthly,x="month_name",y=selected_pollutant,color="station",markers=True,category_orders={"month_name":months_order},color_discrete_sequence=px.colors.qualitative.Bold,template="plotly_dark",title=f"Monthly {selected_pollutant} Levels by Station")
    fig.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",legend=dict(bgcolor="rgba(0,0,0,0)"),xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)",title=f"{selected_pollutant} (\u00b5g/m\u00b3)"))
    st.plotly_chart(fig,use_container_width=True)
    st.markdown(\'<div class="section-header">AQI Heat Calendar</div>\',unsafe_allow_html=True)
    pivot=filtered.groupby(["month","hour"])["AQI"].mean().reset_index()
    pivot_table=pivot.pivot(index="hour",columns="month",values="AQI")
    pivot_table.columns=[months_order[m-1] for m in pivot_table.columns]
    fig2=px.imshow(pivot_table,color_continuous_scale="YlOrRd",template="plotly_dark",aspect="auto",title="AQI Heatmap: Hour of Day vs Month",labels={"color":"AQI"})
    fig2.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6")
    st.plotly_chart(fig2,use_container_width=True)
    st.markdown(\'<div class="section-header">Year-Round Daily AQI</div>\',unsafe_allow_html=True)
    daily_all=filtered.groupby("date")["AQI"].mean().reset_index()
    daily_all["date"]=pd.to_datetime(daily_all["date"])
    daily_all["7d_MA"]=daily_all["AQI"].rolling(7).mean()
    daily_all["30d_MA"]=daily_all["AQI"].rolling(30).mean()
    fig3=go.Figure()
    fig3.add_trace(go.Scatter(x=daily_all["date"],y=daily_all["AQI"],mode="lines",name="Daily AQI",line=dict(color="rgba(255,100,50,0.3)",width=1)))
    fig3.add_trace(go.Scatter(x=daily_all["date"],y=daily_all["7d_MA"],mode="lines",name="7-Day MA",line=dict(color="#ff8c42",width=2)))
    fig3.add_trace(go.Scatter(x=daily_all["date"],y=daily_all["30d_MA"],mode="lines",name="30-Day MA",line=dict(color="#00e5ff",width=2.5)))
    fig3.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",title="Daily AQI Trend with Moving Averages",legend=dict(bgcolor="rgba(0,0,0,0)"),xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
    st.plotly_chart(fig3,use_container_width=True)

with tab2:
    st.markdown(\'<div class="section-header">Pollution Hotspot Map</div>\',unsafe_allow_html=True)
    station_aqi=filtered.groupby("station").agg(AQI=("AQI","mean"),PM25=("PM2.5","mean"),lat=("lat","first"),lon=("lon","first")).reset_index()
    station_aqi["AQI_r"]=station_aqi["AQI"].round(0)
    station_aqi["Category"]=station_aqi["AQI"].apply(aqi_cat)
    fig_map=px.scatter_mapbox(station_aqi,lat="lat",lon="lon",size="AQI",color="AQI",hover_name="station",hover_data={"AQI":":.0f","PM25":":.1f","lat":False,"lon":False},color_continuous_scale="YlOrRd",size_max=50,zoom=10,mapbox_style="carto-darkmatter",title="Delhi Monitoring Stations – AQI Intensity")
    fig_map.update_layout(paper_bgcolor="#0d1526",font_color="#ccd6f6",margin=dict(l=0,r=0,t=40,b=0),height=520)
    st.plotly_chart(fig_map,use_container_width=True)
    station_aqi_sorted=station_aqi.sort_values("AQI",ascending=True)
    colors_bar=[cat_colors.get(aqi_cat(a),"#ff6433") for a in station_aqi_sorted["AQI"]]
    fig_bar=go.Figure(go.Bar(x=station_aqi_sorted["AQI"],y=station_aqi_sorted["station"],orientation="h",marker_color=colors_bar,text=station_aqi_sorted["AQI_r"].astype(str),textposition="outside",textfont=dict(color="#ccd6f6")))
    fig_bar.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",title="Average AQI per Station",xaxis=dict(gridcolor="rgba(255,255,255,0.05)",title="AQI"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"),height=400)
    st.plotly_chart(fig_bar,use_container_width=True)

with tab3:
    st.markdown(\'<div class="section-header">30-Day AQI Forecast</div>\',unsafe_allow_html=True)
    st.info("\U0001f4ca Forecast uses seasonal decomposition + trend extrapolation on historical patterns.")
    daily_all2=df.groupby("date")["AQI"].mean().reset_index()
    daily_all2["date"]=pd.to_datetime(daily_all2["date"])
    daily_all2=daily_all2.sort_values("date")
    last_30=daily_all2.tail(30)["AQI"].values
    trend=np.polyfit(range(len(last_30)),last_30,1)
    future_dates=[daily_all2["date"].max()+timedelta(days=i+1) for i in range(30)]
    base_forecast=[np.polyval(trend,len(last_30)+i) for i in range(30)]
    future_months=[d.month for d in future_dates]
    seasonal_mul={1:1.4,2:1.2,3:1.0,4:0.9,5:0.85,6:0.8,7:0.65,8:0.65,9:0.9,10:1.1,11:1.3,12:1.5}
    forecast_vals=[max(30,b*seasonal_mul.get(m,1)) for b,m in zip(base_forecast,future_months)]
    upper_band=[v+35 for v in forecast_vals]
    lower_band=[max(10,v-35) for v in forecast_vals]
    fig_fc=go.Figure()
    fig_fc.add_trace(go.Scatter(x=daily_all2["date"].tail(90),y=daily_all2["AQI"].tail(90),mode="lines",name="Historical AQI",line=dict(color="#ff8c42",width=2)))
    fig_fc.add_trace(go.Scatter(x=future_dates+future_dates[::-1],y=upper_band+lower_band[::-1],fill="toself",fillcolor="rgba(0,180,255,0.08)",line=dict(color="rgba(0,0,0,0)"),name="Confidence Band"))
    fig_fc.add_trace(go.Scatter(x=future_dates,y=forecast_vals,mode="lines+markers",name="Forecast",line=dict(color="#00e5ff",width=2.5,dash="dash"),marker=dict(size=5)))
    fig_fc.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",title="30-Day AQI Forecast for Delhi",legend=dict(bgcolor="rgba(0,0,0,0)"),xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)",title="AQI"))
    st.plotly_chart(fig_fc,use_container_width=True)
    fc_df=pd.DataFrame({"Date":[d.strftime("%a, %d %b") for d in future_dates[:7]],"Forecast AQI":[int(v) for v in forecast_vals[:7]],"Min AQI":[int(v) for v in lower_band[:7]],"Max AQI":[int(v) for v in upper_band[:7]],"Category":[aqi_cat(v) for v in forecast_vals[:7]]})
    st.dataframe(fc_df,use_container_width=True)

with tab4:
    st.markdown(\'<div class="section-header">Hourly Pollution Patterns</div>\',unsafe_allow_html=True)
    hourly=filtered.groupby(["hour","station"])["AQI"].mean().reset_index()
    fig_h=px.line(hourly,x="hour",y="AQI",color="station",markers=True,template="plotly_dark",color_discrete_sequence=px.colors.qualitative.Pastel,title="Average AQI by Hour of Day")
    fig_h.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",legend=dict(bgcolor="rgba(0,0,0,0)"),xaxis=dict(gridcolor="rgba(255,255,255,0.05)",title="Hour of Day",tickvals=list(range(0,24,2)),ticktext=[f"{h:02d}:00" for h in range(0,24,2)]),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
    st.plotly_chart(fig_h,use_container_width=True)
    col1,col2=st.columns(2)
    with col1:
        filtered2=filtered.copy()
        filtered2["datetime"]=pd.to_datetime(filtered2["datetime"])
        filtered2["weekday"]=filtered2["datetime"].dt.day_name()
        wday_order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        wday=filtered2.groupby("weekday")["AQI"].mean().reindex(wday_order).reset_index()
        fig_w=px.bar(wday,x="weekday",y="AQI",color="AQI",color_continuous_scale="YlOrRd",template="plotly_dark",title="Avg AQI by Weekday")
        fig_w.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",showlegend=False,xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
        st.plotly_chart(fig_w,use_container_width=True)
    with col2:
        sample=filtered.sample(min(3000,len(filtered)),random_state=42)
        fig_s=px.scatter(sample,x="Temperature",y="AQI",color="station",opacity=0.5,template="plotly_dark",color_discrete_sequence=px.colors.qualitative.Bold,title="Temperature vs AQI")
        fig_s.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",legend=dict(bgcolor="rgba(0,0,0,0)"),xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
        st.plotly_chart(fig_s,use_container_width=True)

with tab5:
    st.markdown(\'<div class="section-header">Pollutant Concentration Overview</div>\',unsafe_allow_html=True)
    station_avg=filtered.groupby("station")[pollutants].mean().reset_index()
    fig_radar=go.Figure()
    colors_r=px.colors.qualitative.Bold
    for i,(_,row) in enumerate(station_avg.iterrows()):
        vals=[row[p] for p in pollutants]
        maxes=[filtered[p].max() for p in pollutants]
        norm=[v/m*100 for v,m in zip(vals,maxes)]
        fig_radar.add_trace(go.Scatterpolar(r=norm+[norm[0]],theta=pollutants+[pollutants[0]],mode="lines",fill="toself",line=dict(color=colors_r[i%len(colors_r)],width=2),name=row["station"]))
    fig_radar.update_layout(polar=dict(bgcolor="#0d1526",angularaxis=dict(gridcolor="rgba(255,255,255,0.1)",linecolor="rgba(255,255,255,0.15)",color="#aab8cc"),radialaxis=dict(gridcolor="rgba(255,255,255,0.1)",visible=True,range=[0,100],tickfont=dict(color="#aab8cc"))),paper_bgcolor="#0d1526",font_color="#ccd6f6",title="Pollutant Profile by Station (Normalised)",legend=dict(bgcolor="rgba(0,0,0,0)"),height=500)
    st.plotly_chart(fig_radar,use_container_width=True)
    corr=filtered[pollutants+["AQI","Temperature","Humidity"]].corr()
    fig_corr=px.imshow(corr,text_auto=".2f",color_continuous_scale="RdBu_r",template="plotly_dark",title="Pollutant Correlation Matrix",zmin=-1,zmax=1)
    fig_corr.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",height=450)
    st.plotly_chart(fig_corr,use_container_width=True)
    monthly_box=filtered.copy()
    monthly_box["Month"]=pd.Categorical(monthly_box["month_name"],categories=months_order,ordered=True)
    fig_box=px.box(monthly_box.sort_values("Month"),x="Month",y=selected_pollutant,color="Month",template="plotly_dark",color_discrete_sequence=px.colors.qualitative.Prism,title=f"Monthly Distribution of {selected_pollutant}")
    fig_box.update_layout(plot_bgcolor="#0d1526",paper_bgcolor="#0d1526",font_color="#ccd6f6",showlegend=False,xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
    st.plotly_chart(fig_box,use_container_width=True)

with tab6:
    st.markdown(\'<div class="section-header">Health Advisory for Delhi Residents</div>\',unsafe_allow_html=True)
    if avg_aqi<=100: st.markdown(\'<div class="alert-green">\u2705 <b>Current Air Quality: Acceptable</b> — General population can carry out normal activities.</div>\',unsafe_allow_html=True)
    elif avg_aqi<=200: st.markdown(\'<div class="alert-orange">\u26a0\ufe0f <b>Moderate Pollution</b> — Sensitive groups should reduce prolonged outdoor exposure.</div>\',unsafe_allow_html=True)
    else: st.markdown(\'<div class="alert-red">\U0001f6a8 <b>Hazardous Air Quality</b> — Everyone should minimise outdoor activity.</div>\',unsafe_allow_html=True)
    tips=[("\U0001f637","Wear N95 Masks","N95/KN95 masks filter 95%+ of fine particles. Essential outdoors when AQI > 150."),("\U0001f3e0","Stay Indoors During Peak Hours","7-9 AM and 6-8 PM have the worst pollution. Schedule indoor activities then."),("\U0001f33f","Use Air Purifiers","HEPA air purifiers significantly reduce indoor PM2.5. Run them continuously in bedrooms."),("\U0001f4a7","Stay Hydrated","Drinking 2-3 litres of water daily helps flush pollutants and supports lung function."),("\U0001f697","Avoid Busy Roads","Anand Vihar, ITO intersections have 40% higher pollution. Use metro or alternate routes."),("\U0001fa9f","Keep Windows Closed","During high AQI days, seal windows. Ventilate only during early morning (4-6 AM)."),("\U0001f957","Eat Antioxidant-Rich Foods","Turmeric, ginger, amla and leafy greens protect lungs from oxidative stress."),("\U0001f3c3","Limit Outdoor Exercise","Exercise intensifies breathing. Move workouts indoors when AQI > 100.")]
    cols=st.columns(2)
    for i,(icon,title,text) in enumerate(tips):
        with cols[i%2]: st.markdown(f\'<div class="tip-card"><div class="tip-icon">{icon}</div><div class="tip-title">{title}</div><div class="tip-text">{text}</div></div>\',unsafe_allow_html=True)
    st.markdown(\'<div class="section-header">Long-Term Solutions</div>\',unsafe_allow_html=True)
    solutions=[("\U0001f687","Use Public Transport / Metro","Delhi Metro reduces 400,000+ car trips daily. Expanding ridership cuts NOx by 30%."),("\u26a1","Switch to Electric Vehicles","EVs produce zero tailpipe emissions. Delhi e-vehicle policy offers subsidies up to Rs 1.5 lakh."),("\U0001f331","Plant Trees","Trees absorb CO2 and particulates. Delhi needs green cover above 20%."),("\U0001f3ed","Industrial Emission Controls","Industries in NCR must comply with emission norms. Report violations to DPCC: 1800-11-8787."),("\U0001f525","Stop Stubble Burning","Farm fires contribute 25-40% of Delhi winter pollution. Support bio-decomposer alternatives."),("\u2600\ufe0f","Solar Energy Adoption","Solar rooftop panels reduce coal dependency. MNRE offers 30% subsidy for residential installations.")]
    cols2=st.columns(3)
    for i,(icon,title,text) in enumerate(solutions):
        with cols2[i%3]: st.markdown(f\'<div class="tip-card" style="border-color:rgba(100,255,150,0.2)"><div class="tip-icon">{icon}</div><div class="tip-title" style="color:#7effb2">{title}</div><div class="tip-text">{text}</div></div>\',unsafe_allow_html=True)
    st.markdown("---")
    st.markdown("### Emergency Contacts")
    st.markdown(\'\'\'| Agency | Contact |\n|--------|---------|\n| Delhi Pollution Control Committee (DPCC) | 011-2307-2433 |\n| SAFAR AQI System | safar.tropmet.res.in |\n| Central Pollution Control Board | cpcb.nic.in |\n| AIIMS Emergency (Respiratory) | 011-2658-8500 |\'\'\' )

st.markdown("<br><hr style=\'border-color:rgba(255,100,50,0.2)\'>",unsafe_allow_html=True)
st.markdown(\'<div style="text-align:center;color:#445566;font-size:0.8rem;padding:8px 0">DELHI AIR QUALITY INTELLIGENCE DASHBOARD &middot; Final Year Project &middot; Built with Streamlit + Plotly</div>\',unsafe_allow_html=True)
'''

with open('app.py', 'w') as f:
    f.write(dashboard_code)

print('✅ app.py written successfully!')

In [ ]:
# ── Step 3: Launch with localtunnel (no token needed!) ────────────────────────
import subprocess, threading, time

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)  # wait for streamlit to start

# Use localtunnel to expose port
!npm install -q localtunnel
tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

time.sleep(3)
output = tunnel.stdout.readline().decode().strip()
print('🌐 Dashboard URL:', output)
print('⚠️  If localtunnel asks for a password, visit https://loca.lt/mytunnelpassword to get it.')

---
## 🎉 That's it!
Click the URL above to open your Delhi AQI Dashboard.

### What's inside:
| Tab | Content |
|-----|---------|
| 📈 Trend Analysis | Monthly line charts, AQI heatmap calendar, daily MA trends |
| 🗺️ Hotspot Map | Interactive map of all 12 Delhi monitoring stations |
| 🔮 Forecast | 30-day AQI forecast with confidence bands |
| ⏰ Diurnal Patterns | Hourly & weekday pollution patterns |
| 🧪 Pollutant Breakdown | Radar chart, correlation matrix, boxplots |
| 💡 Health Advisory | Protection tips + long-term solutions |

**Pollutants covered:** PM2.5, PM10, NO₂, SO₂, CO, O₃, NH₃, Temperature, Humidity

**Stations covered:** Anand Vihar, ITO, RK Puram, Punjabi Bagh, Dwarka, Rohini, Okhla, Shadipur, Lodhi Road, Bawana, Mundka, Wazirpur